# **FINANCIAL DERIVATIVE BASICS**

*Here you have the first Jupyter Notebook of your Derivative Pricing course.*
*We will use this format throughout the whole course in order to do a proper mapping of the theoretical concepts in the slides/videos to the hands-on development of the ideas.* 

*In this notebook, we will develop a function in Python in order to construct a simple binomial tree. Make sure you understand this properly because it is going to set the foundation for future work.* 

In [9]:
import os
import sys

# Get the full path of the current notebook
notebook_path = os.path.join(os.getcwd(), sys.argv[0])
print(notebook_path)

c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\ipykernel_launcher.py


Let's start importing some of the libraries that we may need to use down the road:

In [10]:
import numpy as np

In order to compute the binomial tree  in the video/slides of Lesson 1, we need a bunch of input data: upward movement (**u**), downward movement (**d**), risk-free rate (**$r_f$**), time-horizon (**T**), and number of steps in the tree (**N**).

All these would be user-inputs to our function. However, there is one input that we need to consider with caution: the **time-step**. That is, how long does moving from one node to the next one in the tree represent in terms of time?

In our baseline example on the slides the time-horizon for the evolution of the underlying (T) was equal to 1 year. There was 1 step in the tree (N=1). Therefore, there were no doubts that the time-step needed to be equal to 1. 

What if we want a different number of steps, or the time-horizon for the evolution of the underlying is different?
In those cases, the time-step ($dt$) we use for our binomial model will be $→$ $dt = T/N$.

Although we have not yet introduced the utility to specify $dt$, you will realize soon that $dt$ will be helpful for the calculation of risk-neutral probabilities.

## 1. Constructing a Binomial Tree

In the following code snippet, you have the code for a general function to simulate the underlying stock price for some inputs: initial stock price (`S_ini`), time-horizon ($T$), upward ($u$) and downward ($d$) movements, and number of steps (N).

In [11]:
def binomial_tree(S_ini, T, u, d, N):
    S = np.zeros([N + 1, N + 1])  # Underlying price
    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            S[j, i] = S_ini * (u ** (i)) * (d ** (j - i))
    return S

Note that we store everything in a variable, $S$, the one returned by the function. This variable will contain an array with the values of the stock price at each point in time in a lower triangular matrix.

Let's check it by replicating the same tree for $N=2$ that you have in the Lesson 1 slides:

In [12]:
Stock = binomial_tree(100, 1, 1.2, 0.8, 2)
Stock

array([[100.,   0.,   0.],
       [ 80., 120.,   0.],
       [ 64.,  96., 144.]])

## 2. Extending the Tree with Call Option Payoffs

Next, let's extend the previous function by adding another variable that computes the payoffs associated with a Call Option of certain characteristics. Note that we are focusing on a European Call Option with strike price $K=90$, and therefore the payoff is only computed at maturity:

In [13]:
def binomial_tree_call(S_ini, K, T, u, d, N):
    C = np.zeros([N + 1, N + 1])  # Call prices
    S = np.zeros([N + 1, N + 1])  # Underlying price
    for i in range(0, N + 1):
        C[N, i] = max(S_ini * (u ** (i)) * (d ** (N - i)) - K, 0)
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            S[j, i] = S_ini * (u ** (i)) * (d ** (j - i))
    return S, C

It is easy to see that the variable $C$ output by the function will return Call Option payoff at maturity. We can verify this by replicating the simple N=1 tree with Call option payoff from the Lesson 1 slides:

In [14]:
Stock, Call = binomial_tree_call(100, 90, 10, 1.2, 0.8, 10)
print("Underlying Price Evolution:\n", Stock)
print("Call Option Payoff:\n", Call)

Underlying Price Evolution:
 [[100.           0.           0.           0.           0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 80.         120.           0.           0.           0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 64.          96.         144.           0.           0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 51.2         76.8        115.2        172.8          0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 40.96        61.44        92.16       138.24       207.36
    0.           0.           0.           0.           0.
    0.        ]
 [ 32.768       49.152       73.728      110.592      165.888
  248.832        0.           0.           0.           0.
    0.        ]
 [ 26.2144      39.3216      58.9824      88.4736     132.7104
  199.0656     298.5984       0.           0.           0.
    0.        ]
 [ 20.97152     31.45728

## 3. Introducing Risk-Neutral Probabilities and backward induction of Call Option Value

For the final part of this notebook, let's work with the risk-neutral probabilities. Once we have the probabilities, we can, by backward induction, calculate the value of the Call Option (given its future payoffs and the associated probabilities) at each node.

Importantly, once we know the risk-neutral probabilities, the value of the Call Option at each node will depend on the expected payoff in the two potential future scenarios (up or down movements), discounted at risk-free. That is:

$C_{t}= e^{-rdt}[p c_{t+1}^u + (1-p) c_{t+1}^d]$

where $dt$ is the discounted period from one node to the next (**time-step**), and $c_{t+1}^u$ and $c_{t+1}^d$ are the **values of the Call option in the next period**. We will therefore have to start from the last period (maturity) and work backwards, hence the term backward induction.

Note that we use $dt$ here because we are assuming there are a **bunch of periods (steps) in the tree from the initial date until maturity of the option**. Under the 1-step case, we can calculate risk-neutral probabilities as we did in the videos, because $dt=T/N = 1/1 = 1 = T$:

$p=\frac{e^{rT}-d}{u-d}$

Once we consider a different $dt$, we just need to modify $p$ accordingly:

$p=\frac{e^{rdt}-d}{u-d}$

Now, let's write a final function that recognizes all these issues.

In [15]:
def binomial_call_full(S_ini, K, T, r, u, d, N):
    dt = T / N  # Define time step
    p = (np.exp(r * dt) - d) / (u - d)  # Risk neutral probabilities (probs)
    C = np.zeros([N + 1, N + 1])  # Call prices
    S = np.zeros([N + 1, N + 1])  # Underlying price
    for i in range(0, N + 1):
        C[N, i] = max(S_ini * (u ** (i)) * (d ** (N - i)) - K, 0)
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i])
            S[j, i] = S_ini * (u ** (i)) * (d ** (j - i))
    return C[0, 0], C, S

Notice that since we are doing backward induction, the first value of the Call Option Payoff matrix (the last we calculate) is the price of the Call Option today.

Let's replicate with the values from the example in the slides to check it:

In [16]:
call_price, C, S = binomial_call_full(100, 90, 10, 0, 1.2, 0.8, 10)
print("Underlying Price Evolution:\n", S)
print("Call Option Payoff:\n", C)
print("Call Option Price at t=0: ", "{:.2f}".format(call_price))

Underlying Price Evolution:
 [[100.           0.           0.           0.           0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 80.         120.           0.           0.           0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 64.          96.         144.           0.           0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 51.2         76.8        115.2        172.8          0.
    0.           0.           0.           0.           0.
    0.        ]
 [ 40.96        61.44        92.16       138.24       207.36
    0.           0.           0.           0.           0.
    0.        ]
 [ 32.768       49.152       73.728      110.592      165.888
  248.832        0.           0.           0.           0.
    0.        ]
 [ 26.2144      39.3216      58.9824      88.4736     132.7104
  199.0656     298.5984       0.           0.           0.
    0.        ]
 [ 20.97152     31.45728

## 4. Conclusion

In this lesson, we have started to price options in the simple framework of the binomial model. In the next lesson, we will keep working with the binomial model on features like the put-call parity.

See you in the next lesson!